In [0]:
volumes_path = "/Volumes/dbr_dev_ua5816bd/roksolana_shendiu770/raw_files/"

sources = {
    "petroleum_raw.json": "dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_raw",
    "petroleum_prices_raw.json": "dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_prices_raw",
}

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date
from delta.tables import DeltaTable

def ingest_file(file_name, target_table):
    file_path = volumes_path + file_name

    df = (
        spark.read
        .option("multiline", "true")
        .json(file_path)
        .withColumn("source_filename", col("_metadata.file_path"))
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
        .dropDuplicates(["period", "series"])
    )

    if spark.catalog.tableExists(target_table):
        delta_table = DeltaTable.forName(spark, target_table)
        (
            delta_table.alias("target")
            .merge(
                df.alias("source"),
                "target.period = source.period AND target.series = source.series"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"[{target_table}] Merge completed.")
    else:
        df.write.format("delta").saveAsTable(target_table)
        print(f"[{target_table}] Created table and loaded {df.count()} rows.")

In [0]:
for file_name, target_table in sources.items():
    ingest_file(file_name, target_table)

In [0]:
for target_table in sources.values():
    count = spark.table(target_table).count()
    print(f"{target_table}: {count} rows")

In [0]:
for file_name, target_table in sources.items():
    ingest_file(file_name, target_table)